# Pergunta 4 — Perfil dos municípios que atingiram a meta do IDEB de 6.0

Quais municípios de Alagoas já atingiram a meta nacional do IDEB de 6.0
no Ensino Fundamental Anos Iniciais, e o que os diferencia dos demais?

**Perguntas específicas:**
- Quantos municípios atingiram a meta em 2023?
- Em que ano cada um cruzou a linha de 6.0 pela primeira vez?
- Eles têm perfil de infraestrutura diferente dos que não atingiram?
- Estavam entre os que mais melhoraram entre 2005 e 2023?

**Input:** `data/processed/ideb_series_al.parquet`

In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, str(Path().resolve().parent))
from src.config import IDEB_SERIES_PARQUET

META = 6.0

df = pd.read_parquet(IDEB_SERIES_PARQUET)
ai = df[df["etapa"] == "EF Anos Iniciais"].copy()

print(f"Municípios: {ai['CO_MUNICIPIO'].nunique()}")
print(f"Anos disponíveis: {sorted(ai['ano'].unique())}")

Municípios: 102
Anos disponíveis: [np.int64(2005), np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2017), np.int64(2019), np.int64(2021), np.int64(2023)]


## Quantos municípios atingiram a meta em 2023?

In [2]:
ideb_2023 = (
    ai[ai["ano"] == 2023]
    [["CO_MUNICIPIO", "NO_MUNICIPIO", "ideb"]]
    .rename(columns={"ideb": "ideb_2023"})
    .sort_values("ideb_2023", ascending=False)
    .reset_index(drop=True)
)

acima = ideb_2023[ideb_2023["ideb_2023"] >= META]
abaixo = ideb_2023[ideb_2023["ideb_2023"] < META]

print(f"Total com IDEB 2023 disponível: {len(ideb_2023)}")
print(f"Atingiram a meta (≥ {META}): {len(acima)} municípios")
print(f"Ainda abaixo da meta: {len(abaixo)} municípios")
print(f"\nMunicípios que atingiram a meta em 2023:")
print(acima.to_string(index=False))

Total com IDEB 2023 disponível: 100
Atingiram a meta (≥ 6.0): 31 municípios
Ainda abaixo da meta: 69 municípios

Municípios que atingiram a meta em 2023:
CO_MUNICIPIO          NO_MUNICIPIO  ideb_2023
     2708105     Santana do Mundaú        9.8
     2702306              Coruripe        9.7
     2709301    União dos Palmares        9.7
     2703007            Ibateguara        9.6
     2709152       Teotônio Vilela        9.0
     2703759       Jequiá da Praia        8.9
     2704005             Junqueiro        8.4
     2708303      São José da Laje        8.3
     2701100            Branquinha        8.3
     2707503 Porto Real do Colégio        7.5
     2701407          Campo Alegre        7.3
     2701506          Campo Grande        7.0
     2702702         Feliz Deserto        7.0
     2705507                Murici        6.9
     2702504          Dois Riachos        6.9
     2702009         Coité do Nóia        6.6
     2704203    Limoeiro de Anadia        6.6
     2708204      

In [3]:
# Para cada município que está acima da meta em 2023,
# encontrar o primeiro ano em que IDEB >= 6.0
primeiro_ano = []

for cod in acima["CO_MUNICIPIO"]:
    serie = ai[ai["CO_MUNICIPIO"] == cod].sort_values("ano")
    cruzou = serie[serie["ideb"] >= META]
    if not cruzou.empty:
        primeiro_ano.append({
            "CO_MUNICIPIO": cod,
            "NO_MUNICIPIO": serie["NO_MUNICIPIO"].iloc[0],
            "ideb_2023": acima[acima["CO_MUNICIPIO"] == cod]["ideb_2023"].values[0],
            "primeiro_ano_meta": cruzou["ano"].min(),
        })

df_meta = pd.DataFrame(primeiro_ano).sort_values("primeiro_ano_meta")

print("Municípios que atingiram a meta — por ano de cruzamento:")
print(df_meta.to_string(index=False))

Municípios que atingiram a meta — por ano de cruzamento:
CO_MUNICIPIO          NO_MUNICIPIO  ideb_2023  primeiro_ano_meta
     2702306              Coruripe        9.7               2015
     2703759       Jequiá da Praia        8.9               2015
     2701407          Campo Alegre        7.3               2015
     2709152       Teotônio Vilela        9.0               2017
     2704005             Junqueiro        8.4               2017
     2707800               Roteiro        6.1               2017
     2701100            Branquinha        8.3               2019
     2708105     Santana do Mundaú        9.8               2019
     2708303      São José da Laje        8.3               2019
     2709301    União dos Palmares        9.7               2021
     2705507                Murici        6.9               2021
     2703007            Ibateguara        9.6               2021
     2708204              São Brás        6.6               2021
     2702108    Colônia Leopoldin

In [4]:
fig = go.Figure()

# Linha de cada município que atingiu a meta
for cod in acima["CO_MUNICIPIO"]:
    serie = ai[ai["CO_MUNICIPIO"] == cod].sort_values("ano")
    nome = serie["NO_MUNICIPIO"].iloc[0]
    fig.add_trace(go.Scatter(
        x=serie["ano"], y=serie["ideb"],
        mode="lines+markers",
        name=nome,
        line=dict(width=1.5),
        marker=dict(size=5),
        hovertemplate=f"<b>{nome}</b><br>Ano: %{{x}}<br>IDEB: %{{y:.1f}}<extra></extra>",
    ))

# Linha da meta
fig.add_hline(
    y=META, line_dash="dash", line_color="#D85A30",
    annotation_text="Meta 6.0",
    annotation_position="right",
)

fig.update_layout(
    title="Evolução do IDEB — municípios que atingiram a meta de 6.0",
    xaxis_title="Ano",
    yaxis_title="IDEB",
    xaxis=dict(tickmode="array", tickvals=sorted(ai["ano"].unique())),
    height=480,
    legend=dict(
        orientation="v", x=1.02, y=1,
        font=dict(size=9),
    ),
)
fig.show()

In [5]:
# Carregar dados de infraestrutura do notebook 03
# (recriar o df_infra_mun aqui para não depender do outro notebook)
from pathlib import Path
from src.config import DATA_INEP
import pandas as pd

censo_path = (
    DATA_INEP
    / "microdados_censo_escolar_2023"
    / "dados"
    / "microdados_ed_basica_2023.csv"
)

COLUNAS_INFRA = [
    "CO_UF", "CO_MUNICIPIO", "TP_DEPENDENCIA",
    "TP_SITUACAO_FUNCIONAMENTO",
    "IN_BIBLIOTECA", "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_LABORATORIO_INFORMATICA", "IN_LABORATORIO_CIENCIAS",
    "IN_QUADRA_ESPORTES", "IN_INTERNET",
]

ITENS = [
    "IN_BIBLIOTECA", "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_LABORATORIO_INFORMATICA", "IN_LABORATORIO_CIENCIAS",
    "IN_QUADRA_ESPORTES", "IN_INTERNET",
]

chunks = []
for chunk in pd.read_csv(
    censo_path, sep=";", encoding="latin-1",
    usecols=COLUNAS_INFRA, chunksize=10_000,
):
    f = chunk[
        (chunk["CO_UF"] == 27) &
        (chunk["TP_SITUACAO_FUNCIONAMENTO"] == 1) &
        (chunk["TP_DEPENDENCIA"].isin([2, 3]))
    ]
    if not f.empty:
        chunks.append(f)

df_censo = pd.concat(chunks, ignore_index=True)

df_infra = (
    df_censo.groupby("CO_MUNICIPIO")[ITENS]
    .mean().multiply(100).round(1).reset_index()
)
df_infra["CO_MUNICIPIO"] = df_infra["CO_MUNICIPIO"].astype(str)

# Merge com classificação acima/abaixo da meta
ideb_2023["grupo"] = ideb_2023["ideb_2023"].apply(
    lambda x: f"Acima da meta (≥{META})" if x >= META else f"Abaixo da meta (<{META})"
)

df_comp = df_infra.merge(
    ideb_2023[["CO_MUNICIPIO", "grupo"]], on="CO_MUNICIPIO", how="inner"
)

comparacao = (
    df_comp.groupby("grupo")[ITENS]
    .mean().round(1).T
    .reset_index()
    .rename(columns={"index": "item"})
)

print("Cobertura média de infraestrutura por grupo (%):")
print(comparacao.to_string(index=False))

Cobertura média de infraestrutura por grupo (%):
                      item  Abaixo da meta (<6.0)  Acima da meta (≥6.0)
             IN_BIBLIOTECA                   17.1                  19.1
IN_BIBLIOTECA_SALA_LEITURA                   34.2                  34.4
IN_LABORATORIO_INFORMATICA                   15.9                  19.8
   IN_LABORATORIO_CIENCIAS                    9.3                   5.6
        IN_QUADRA_ESPORTES                   21.2                  16.5
               IN_INTERNET                   91.2                  93.0


In [6]:
NOMES = {
    "IN_BIBLIOTECA": "Biblioteca",
    "IN_BIBLIOTECA_SALA_LEITURA": "Biblioteca/sala leitura",
    "IN_LABORATORIO_INFORMATICA": "Lab. informática",
    "IN_LABORATORIO_CIENCIAS": "Lab. ciências",
    "IN_QUADRA_ESPORTES": "Quadra esportiva",
    "IN_INTERNET": "Internet",
}

comparacao["item_label"] = comparacao["item"].map(NOMES)

fig = go.Figure()

fig.add_trace(go.Bar(
    name=f"Acima da meta (≥{META}) — {len(acima)} municípios",
    x=comparacao["item_label"],
    y=comparacao[f"Acima da meta (≥{META})"],
    marker_color="#1D9E75",
))

fig.add_trace(go.Bar(
    name=f"Abaixo da meta (<{META}) — {len(abaixo)} municípios",
    x=comparacao["item_label"],
    y=comparacao[f"Abaixo da meta (<{META})"],
    marker_color="#D85A30",
))

fig.update_layout(
    title="Infraestrutura média: municípios acima vs abaixo da meta IDEB 6.0",
    yaxis_title="% de escolas com o item",
    barmode="group",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

### Interpretação

**31 de 100 municípios (31%) já atingiram a meta de 6.0 em 2023**,
um resultado expressivo considerando que em 2005 nenhum município
alagoano superava 4.0 no EF Anos Iniciais.

**A progressão não foi linear — aconteceu em ondas:**
Os primeiros 3 municípios cruzaram a meta em 2015 (Coruripe, Jequiá da
Praia e Campo Alegre). O ritmo se manteve estável até 2021, quando a
pandemia não impediu que mais 5 municípios atingissem a meta. O salto
mais expressivo foi em **2023, quando 17 municípios cruzaram a linha de
6.0 de uma vez** — sugerindo uma aceleração sistêmica na última edição.

**Infraestrutura não diferencia os grupos:**
Os municípios acima da meta não têm cobertura de infraestrutura
significativamente maior que os abaixo. Em dois itens — laboratório de
ciências e quadra esportiva — os municípios acima da meta têm até
**menos** cobertura que os abaixo (5.6% vs 9.3% e 16.5% vs 21.2%).

Este achado reforça a conclusão da pergunta 3: **em Alagoas, a presença
de infraestrutura física não é o fator determinante do desempenho
escolar.** Outros fatores — qualidade da gestão pedagógica, programas
de alfabetização, engajamento comunitário — explicam melhor por que
alguns municípios ultrapassaram a meta enquanto outros com infraestrutura
similar ainda não chegaram lá.

**Os 69 municípios ainda abaixo da meta** têm IDEB médio de 5.3 em 2023
— distância de apenas 0.7 pontos da meta. Se a tendência de aceleração
de 2023 se mantiver, é plausível que uma parcela significativa deles
atinja 6.0 na próxima edição (2025).

In [11]:
municipios_dupla = composicao[composicao["tipo"] == "Estadual + Municipal"]["CO_MUNICIPIO"].tolist()

nomes = df_al[["CO_MUNICIPIO", "NO_MUNICIPIO"]].drop_duplicates().set_index("CO_MUNICIPIO")

ideb_pub = df_al[
    (df_al["CO_MUNICIPIO"].isin(municipios_dupla)) &
    (df_al["REDE"] == "Pública")
][["CO_MUNICIPIO", "NO_MUNICIPIO", "VL_OBSERVADO_2023"]].rename(
    columns={"VL_OBSERVADO_2023": "ideb_publico"}
)

ideb_est = df_al[
    (df_al["CO_MUNICIPIO"].isin(municipios_dupla)) &
    (df_al["REDE"] == "Estadual")
][["CO_MUNICIPIO", "VL_OBSERVADO_2023"]].rename(
    columns={"VL_OBSERVADO_2023": "ideb_estadual"}
)

ideb_mun = df_al[
    (df_al["CO_MUNICIPIO"].isin(municipios_dupla)) &
    (df_al["REDE"] == "Municipal")
][["CO_MUNICIPIO", "VL_OBSERVADO_2023"]].rename(
    columns={"VL_OBSERVADO_2023": "ideb_municipal"}
)

df_dupla = (
    ideb_pub
    .merge(ideb_est, on="CO_MUNICIPIO", how="left")
    .merge(ideb_mun, on="CO_MUNICIPIO", how="left")
)

print("Municípios com redes Estadual e Municipal em 2023:")
print(df_dupla.to_string(index=False))

Municípios com redes Estadual e Municipal em 2023:
CO_MUNICIPIO        NO_MUNICIPIO  ideb_publico  ideb_estadual  ideb_municipal
     2704302              Maceió           5.3            5.4             5.3
     2706307 Palmeira dos Índios           5.6            5.5             5.7
     2706422           Pariconha           5.3            4.1             5.7
     2706703              Penedo           6.4            6.8             6.3
     2708006  Santana do Ipanema           4.9            5.1             4.8


### Observação — composição de redes por município

95 dos 100 municípios com IDEB 2023 disponível possuem apenas rede
municipal de EF Anos Iniciais. Apenas 5 municípios têm simultaneamente
redes estadual e municipal: Maceió, Palmeira dos Índios, Pariconha,
Penedo e Santana do Ipanema.

Com uma amostra de apenas 5 municípios na categoria "duas redes",
não é possível fazer inferências estatísticas sobre o efeito da
composição de redes no desempenho. O que os dados mostram
qualitativamente é que o IDEB Pública nesses municípios fica entre
os valores das duas redes separadas — confirmando que é uma média
ponderada pelo número de alunos de cada rede.

O caso mais discrepante é Pariconha, onde a rede estadual (4.1) e
municipal (5.7) diferem em 1.6 pontos no mesmo município.